# Shared data preparation — building the group's master dataset

This notebook documents the data-engineering steps used to go from raw lab exports to the clean, merged dataset used across the group's analyses. I contributed this pipeline alongside teammates. It's included here for completeness/transparency, not as one of my two analysis questions.

**Note on reproducibility**: this operates on raw folders (raw per-sample measurement files, ~850 of them) that aren't included in this repo due to size — see the README. The final outputs of this pipeline (`Mastersheet.xlsx`, `merged_data.xlsx`) *are* included, so Questions 2 and 5 can be run without re-running this notebook.

**Stages**:
1. Standardise raw filenames and convert CSV exports to XLSX
2. Filter the master metadata sheet to samples with valid measurements, twice (an initial pass and a refined pass)
3. Match individual fossil sample files to valid metadata using filename parsing, and copy matches into a clean folder
4. Standardise column names across all sample files
5. Merge the final mastersheet with environmental (climate proxy) data by nearest geological age

## Setup

In [ ]:
from pathlib import Path

DATA_DIR = Path("../data")


In [ ]:
import os
import pandas as pd
import re
import shutil
from collections import defaultdict
from IPython.display import display

## Stage 1 — Standardise raw filenames, convert CSV → XLSX

Raw lab exports had an inconsistent `"Count and Measure of "` filename prefix, and some were CSV rather than XLSX. This strips the prefix and converts formats for consistency.

In [ ]:
folder_path = DATA_DIR / "raw" / "925 Data additional data"

In [ ]:
# Prefix to remove
prefix = "Count and Measure of "

In [ ]:
# Iterate through all files in the folder
for filename in os.listdir(folder_path):
    if filename.startswith(prefix):
        new_name = filename[len(prefix):]  # Remove the prefix
        old_path = os.path.join(folder_path, filename)
        new_path = os.path.join(folder_path, new_name)
        
        # Rename the file
        os.rename(old_path, new_path)
        print(f"Renamed: {filename} → {new_name}")

print("All files have been renamed!")

In [ ]:
# Define source and destination folders
csv_source_folder = DATA_DIR / "raw" / "925 Data additional data"  # Folder containing CSV files
xlsx_destination_folder = DATA_DIR / "raw" / "Ceara Rise data"  # Final folder where XLSX files will be stored

In [ ]:
# Ensure the destination folder exists
os.makedirs(xlsx_destination_folder, exist_ok=True)

In [ ]:
# Convert CSV to XLSX and save directly in the final destination
for filename in os.listdir(csv_source_folder):
    if filename.endswith(".csv"):
        csv_path = os.path.join(csv_source_folder, filename)

        # Read CSV file
        df = pd.read_csv(csv_path)

        # Create XLSX filename
        new_filename = filename.replace(".csv", ".xlsx")
        xlsx_path = os.path.join(xlsx_destination_folder, new_filename)

        # Save as XLSX in final destination
        df.to_excel(xlsx_path, index=False, engine="openpyxl")

        print(f"Converted & Saved: {filename} → {xlsx_path}")

print("All CSV files have been converted to XLSX and saved in the final destination!")

## Stage 2 — Filter the mastersheet to valid measurements (first pass)

The raw mastersheet includes rows without a completed diameter measurement — these are dropped so only samples with usable morphometric data remain.

In [ ]:
# Loading and Filtering the Master Sheet
master_file_path = DATA_DIR / "raw" / "925_Mastersheet_filled 1.xlsx"
df_master = pd.read_excel(master_file_path, sheet_name="Request_925_all WORKING SAMPLE ", header=1)

In [ ]:
# Remove rows where 'Size.Mean.DiameterMean' is missing
df_filtered_master = df_master.dropna(subset=['Size.Mean.DiameterMean'])

In [ ]:
# Saving the filtered master sheet to a new Excel file
filtered_master_file = DATA_DIR / "filtered_mastersheet.xlsx"
df_filtered_master.to_excel(filtered_master_file, index=False, engine="openpyxl")
print(f"Filtered master sheet saved as '{filtered_master_file}'.")

## Stage 3 — Match individual sample files to valid metadata (first pass)

Each raw sample file's name encodes its site/hole/core/section. We parse that out with a regex, build a lookup set of metadata combinations present in the filtered mastersheet, and copy across only the sample files that have a valid match — i.e. discard files that don't correspond to a usable mastersheet row.

In [ ]:
# Building a Set of Valid Metadata from the Filtered Master Sheet
metadata_set = set()
for _, row in df_filtered_master.iterrows():
    try:
        key = (
            int(row['SITE']),
            str(row['HOLE']).strip(),
            int(row['CORE']),
            int(row['SECTION']),
            int(row['TOP_DEPTH']),
            int(row['BOTTOM_DEPTH'])
        )
        metadata_set.add(key)
    except Exception as e:
        print(f"Error processing master row: {e}")

print(f"Number of valid metadata entries in filtered master sheet: {len(metadata_set)}")

In [ ]:
# Defining Function to Extract Metadata from File Names
def extract_metadata(filename):
    pattern = r'(\d+)([A-Z]),\s*(\d+)([A-Z])-(\d+),\s*(\d+)-(\d+)'
    match = re.search(pattern, filename)
    if match:
        try:
            site = int(match.group(1))
            hole = match.group(2)
            core = int(match.group(3))
            # match.group(4) is CORE_TYPE which we are not using for matching because it's same
            section = int(match.group(5))
            top_depth = int(match.group(6))
            bottom_depth = int(match.group(7))
            return (site, hole, core, section, top_depth, bottom_depth)
        except Exception as e:
            print(f"Error extracting metadata from {filename}: {e}")
            return None
    else:
        return None

In [ ]:
# Copying Matching Fossil Files to a New Folder
data_folder = DATA_DIR / "raw" / "Ceara Rise data"  # Folder containing individual fossil files
new_folder = DATA_DIR / "raw" / "Updated Ceara Rise Data"  # New folder to save the matches
os.makedirs(new_folder, exist_ok=True)

for file_name in os.listdir(data_folder):
    if file_name.endswith(".xlsx"):
        file_path = os.path.join(data_folder, file_name)
        meta = extract_metadata(file_name)
        if meta is None:
            print(f"Could not extract metadata from '{file_name}'. Skipping file.")
            continue
        if meta in metadata_set:
            try:
                shutil.copy(file_path, new_folder)
                print(f"Copied file '{file_name}' to '{new_folder}' (metadata {meta} found).")
            except Exception as e:
                print(f"Error copying file '{file_name}': {e}")
        else:
            print(f"File '{file_name}' skipped (metadata {meta} not found in filtered master sheet).")

## Stage 4 — Clean and standardise columns across sample files

Raw sample files had inconsistent column naming (e.g. unit suffixes like `areaµm`). These steps clean/standardise columns across the whole matched-file set.

In [ ]:
def clean_excel_files(folder_path):
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".xlsx"):
            file_path = os.path.join(folder_path, file_name)
            
            # Read the Excel file
            df = pd.read_excel(file_path)
            
            # Check the last three rows to see if they contain summary statistics
            last_three = df.tail(3)
            if any(last_three.iloc[:, 0].astype(str).str.contains("Count in filter ranges|Mean|Standard Deviation", na=False)):
                
                # Remove the last three rows
                df_cleaned = df.iloc[:-3]
                
                # Save the cleaned file back to the same location
                df_cleaned.to_excel(file_path, index=False)
                print(f"Cleaned: {file_name}")
            else:
                print(f"Skipped: {file_name} (No summary statistics found)")

In [ ]:
FOLDER_PATH = DATA_DIR / "raw" / "Updated Ceara Rise Data"
clean_excel_files(FOLDER_PATH)

In [ ]:
def filter_columns(folder_path):
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".xlsx"):
            file_path = os.path.join(folder_path, file_name)
            
            # Read the Excel file
            df = pd.read_excel(file_path)
            
            # Standardize column names by making them lowercase and removing spaces
            df.columns = df.columns.str.lower().str.replace(" ", "").str.replace(".", "")
            
            # Possible variations of 'Object ID'
            possible_object_id_names = ['objectid', 'object_id', 'object.id']
            
            # Find the first matching 'Object ID' column
            for col in df.columns:
                if any(name in col for name in possible_object_id_names):
                    object_id_index = df.columns.get_loc(col)
                    df_filtered = df.iloc[:, object_id_index:]
                    
                    # Save the filtered file back to the same location
                    df_filtered.to_excel(file_path, index=False)
                    print(f"Filtered columns for: {file_name}")
                    break
            else:
                print(f"Skipped: {file_name} (No recognized 'Object ID' column found)")

In [ ]:
FOLDER_PATH = DATA_DIR / "raw" / "Updated Ceara Rise Data"
filter_columns(FOLDER_PATH)

## Stage 5 — Second filtering pass, using the now-cleaned data

With cleaned sample files in hand, we re-filter the mastersheet (this time on area rather than diameter) and re-match sample files, producing the final `Mastersheet.xlsx` used throughout the rest of the project.

In [ ]:
# Loading and Filtering the Master Sheet
master_file_path = DATA_DIR / "filtered_mastersheet.xlsx"
df_master = pd.read_excel(master_file_path, sheet_name="Sheet1", header=0)

In [ ]:
# Remove rows where 'Size.Mean.Area' is missing
df_filtered_master = df_master.dropna(subset=['Size.Mean.Area'])

In [ ]:
# Saving the filtered master sheet to a new Excel file
filtered_master_file = DATA_DIR / "Mastersheet.xlsx"
df_filtered_master.to_excel(filtered_master_file, index=False, engine="openpyxl")
print(f"Filtered master sheet saved as '{filtered_master_file}'.")

In [ ]:
# Building a Set of Valid Metadata from the Filtered Master Sheet
metadata_set = set()
for _, row in df_filtered_master.iterrows():
    try:
        key = (
            int(row['SITE']),
            str(row['HOLE']).strip(),
            int(row['CORE']),
            int(row['SECTION']),
            int(row['TOP_DEPTH']),
            int(row['BOTTOM_DEPTH'])
        )
        metadata_set.add(key)
    except Exception as e:
        print(f"Error processing master row: {e}")

print(f"Number of valid metadata entries in filtered master sheet: {len(metadata_set)}")

In [ ]:
# Defining Function to Extract Metadata from File Names
def extract_metadata(filename):
    pattern = r'(\d+)([A-Z]),\s*(\d+)([A-Z])-(\d+),\s*(\d+)-(\d+)'
    match = re.search(pattern, filename)
    if match:
        try:
            site = int(match.group(1))
            hole = match.group(2)
            core = int(match.group(3))
            # match.group(4) is CORE_TYPE which we are not using for matching because it's same
            section = int(match.group(5))
            top_depth = int(match.group(6))
            bottom_depth = int(match.group(7))
            return (site, hole, core, section, top_depth, bottom_depth)
        except Exception as e:
            print(f"Error extracting metadata from {filename}: {e}")
            return None
    else:
        return None

In [ ]:
# Copying Matching Fossil Files to a New Folder
data_folder = DATA_DIR / "raw" / "Updated Ceara Rise data"
new_folder = DATA_DIR / "raw" / "Final Ceara Rise Data"
os.makedirs(new_folder, exist_ok=True)

for file_name in os.listdir(data_folder):
    if file_name.endswith(".xlsx"):
        file_path = os.path.join(data_folder, file_name)
        meta = extract_metadata(file_name)
        if meta is None:
            print(f"Could not extract metadata from '{file_name}'. Skipping file.")
            continue
        if meta in metadata_set:
            try:
                shutil.copy(file_path, new_folder)
                print(f"Copied file '{file_name}' to '{new_folder}' (metadata {meta} found).")
            except Exception as e:
                print(f"Error copying file '{file_name}': {e}")
        else:
            print(f"File '{file_name}' skipped (metadata {meta} not found in filtered master sheet).")


## Stage 6 — Column-name audit

A quick check to see what column-name variations still exist across the final sample file set, and standardise them to a consistent naming scheme.

In [ ]:
# Set the folder path where all your files are stored
folder_path = DATA_DIR / "raw" / "Final Ceara Rise Data"
output_file = folder_path / "column_variations.xlsx"

In [ ]:
# Dictionary to store column variations and their occurrence count
column_variations = defaultdict(int)

In [ ]:
# Get a list of all Excel files in the folder
files = [f for f in os.listdir(folder_path) if f.endswith(".xlsx")]

In [ ]:
# Iterate through each file and extract column names
for file in files:
    file_path = os.path.join(folder_path, file)
    
    try:
        df = pd.read_excel(file_path, nrows=1)  # Read only header row for efficiency
        col_tuple = tuple(sorted(df.columns))  # Sort column names to avoid order affecting uniqueness
        column_variations[col_tuple] += 1
    except Exception as e:
        print(f"Error reading {file}: {e}")

In [ ]:
# Convert results to a DataFrame for easy viewing
column_variations_df = pd.DataFrame(
    [(list(cols), count) for cols, count in column_variations.items()],
    columns=["Column Names", "Count"]
)

In [ ]:
# Save to an Excel file
column_variations_df.to_excel(output_file, index=False)

In [ ]:
# Display DataFrame in Jupyter Notebook
display(column_variations_df)
print(f"Column variations saved to: {output_file}")

In [ ]:
folder_path = DATA_DIR / "raw" / "Final Ceara Rise Data"

In [ ]:
# Define the column renaming mapping (excluding columns that should remain untouched)
column_mapping = {
    "areaµm": "area",
    "areaµm²": "area",
    "area(µm²)": "area",
    "max(diameter)(µm)": "max(diameter)",
    "maxdiameterµm": "max(diameter)",
    "mean(diameter)(µm)": "mean(diameter)",
    "meandiameterµm": "mean(diameter)",
    "min(diameter)(µm)": "min(diameter)",
    "mindiameterµm": "min(diameter)",
    "mean(grayintensityvalue)": "mean(gray intensity value)",
    "meangrayintensityvalue": "mean(gray intensity value)"
}

In [ ]:
# Define variations of "perimeter" that should be renamed if necessary
perimeter_variations = {"perimeter(µm)", "perimeterµm"}

In [ ]:
# Columns that should remain unchanged
untouched_columns = {"shapefactor", "sphericity", "elongation", "objectid"}

In [ ]:
# Get all Excel files in the folder
files = [f for f in os.listdir(folder_path) if f.endswith(".xlsx") and f != "column_variations.xlsx"]

In [ ]:
# Iterate through each file and rename columns
for file in files:
    file_path = os.path.join(folder_path, file)
    
    try:
        df = pd.read_excel(file_path)  # Read full data
        original_columns = set(df.columns)  # Keep track of original columns

        # Handle "perimeter" renaming separately
        if "perimeter" not in original_columns:  # Only rename if "perimeter" does not already exist
            for var in perimeter_variations:
                if var in original_columns:
                    df.rename(columns={var: "perimeter"}, inplace=True)
                    break  # Rename only one occurrence

        # Rename other columns based on mapping, ensuring untouched columns remain unchanged
        df.rename(columns={col: new_col for col, new_col in column_mapping.items() if col in original_columns and col not in untouched_columns}, inplace=True)

        # Save back to Excel (overwrite original)
        df.to_excel(file_path, index=False)
        print(f"Updated {file} with standardized column names.")
    
    except Exception as e:
        print(f"Error processing {file}: {e}")

In [ ]:
print("All column names have been standardized across the files, with required columns untouched.")

## Stage 7 — Merge with environmental (climate proxy) data

Finally, merge the mastersheet with the benthic isotope environmental dataset, matching each fossil sample to the environmental record with the nearest geological age (since the two datasets weren't sampled at identical time points). This produces `merged_data.xlsx`, the input used in Question 5.

In [ ]:
# === Load files ===
master = pd.read_excel(DATA_DIR / "Mastersheet.xlsx")
env = pd.read_excel(DATA_DIR / "raw" / "environmental_data.xlsx")

# === Extract and convert age columns to a common numeric format ===
master['Age_Ma'] = pd.to_numeric(master['Age (Ma)'], errors='coerce')
env['Age_Ma'] = pd.to_numeric(env['age_tuned'], errors='coerce')

# === Sort by age (required for merge_asof) ===
master_sorted = master.sort_values('Age_Ma').dropna(subset=['Age_Ma'])
env_sorted = env.sort_values('Age_Ma').dropna(subset=['Age_Ma'])

# === Nearest-neighbour merge on geological age ===
merged = pd.merge_asof(master_sorted, env_sorted, on='Age_Ma', direction='nearest')

output_path = DATA_DIR / "merged_data.xlsx"
merged.to_excel(output_path, index=False)